#小项目"术语解释器"+周复盘

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import os

load_dotenv()

#1.定义输出结构
class TermExplanation(BaseModel):
    term:str = Field(description="术语名称")
    one_sentence:str = Field(description="一句话解释")
    detail:str = Field(description="详细解释，2-3句话")
    example:str = Field(description="一个代码或生活类比示例")

#2.解析器+模板
parser = PydanticOutputParser(pydantic_object=TermExplanation)
prompt = ChatPromptTemplate.from_messages([("system","你是资深编程导师。严格按下面的格式输出：\n{format_instructions}"),("user","请解释编程术语：{term}")]).partial(format_instructions=parser.get_format_instructions())

#3.模型
llm = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
    temperature=0.3,
    max_tokens=500
)

#4.管道
chain = prompt|llm|parser

#5.主循环
print("术语解释器v1(输入exit退出)")

while True:
    term = input("请输入术语:").strip()
    if term.lower() == "exit":
        break
    try:
        exp = chain.invoke({"term":term})
        print(f"\n术语:{exp.term}")
        print(f"一句话:{exp.one_sentence}")
        print(f"详解:{exp.detail}")
        print(f"示例:{exp.example}\n")
    except Exception as e:
        print(f"这次没解析成功,换个问法试试。错误:{e}\n")

术语解释器v1(输入exit退出)

术语:LangGraph
一句话:LangGraph 是一个用于构建有状态、可编排的 AI 代理工作流的库，它将复杂逻辑建模为图结构。
详解:LangGraph 基于图的概念，其中节点表示计算步骤或函数，边表示数据流或控制流。它特别适用于需要循环、分支和持久化状态的 AI 应用（如多轮对话、工具调用代理），开发者可以精细控制每一步的执行顺序和状态更新。相比简单的链式调用，LangGraph 能更灵活地处理动态决策和复杂交互。
示例:想象一个客服机器人流程：先判断用户意图，若需要查询订单则进入订单节点，否则进入常见问题节点；每次对话后保存状态，以便下次继续。在 LangGraph 中，这些决策点就是图中的边，每个处理步骤就是节点。

